# 🩺 NetraAI (SIH26038) — APTOS 2019 Blindness Detection Suite
### Fast Kaggle / Google Colab GPU Training Notebook (Zero Drive Upload Needed)

This notebook trains an **EfficientNet-B3 Deep Convolutional Neural Network** on the **3,662 high-resolution scans** of the **APTOS 2019 Blindness Detection** dataset.

⚡ **Zero Google Drive Upload Needed**:
- **Option 1 (Fastest — 0 sec setup)**: Run directly inside a free **Kaggle Notebook** with GPU P100 (dataset is pre-mounted).
- **Option 2 (Google Colab)**: Auto-downloads dataset from Kaggle servers in ~45 seconds via Kaggle API.

💾 **Output**: `grading_efficientnet_b3.pt` (Directly loads into NetraAI `ml/checkpoints/`)

--- 
## 1. Hardware Verification (GPU P100 / T4)

In [ ]:
!nvidia-smi

import os
import sys
import glob
import random
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score, confusion_matrix, classification_report

# Set deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Training on Hardware Device: {device}")

--- 
## 2. Dataset Detection: Kaggle Native vs. Colab Fast Download

The cell below auto-detects whether you are running inside **Kaggle** or **Google Colab**.

In [ ]:
# Detect environment
IS_KAGGLE = os.path.exists('/kaggle/input/aptos2019-blindness-detection')
DATASET_DIR = None

if IS_KAGGLE:
    print("✅ Running natively inside Kaggle!")
    DATASET_DIR = '/kaggle/input/aptos2019-blindness-detection'
    OUTPUT_DIR = '/kaggle/working'
else:
    print("🌐 Running inside Google Colab.")
    OUTPUT_DIR = '/content'
    DATASET_DIR = '/content/dataset'
    
    if not os.path.exists(os.path.join(DATASET_DIR, 'train.csv')):
        print("\n📥 Downloading APTOS 2019 directly from Kaggle via API (takes ~45s) ...")
        # Check for kaggle.json or credentials
        if not os.path.exists('/root/.kaggle/kaggle.json'):
            print("Please provide your Kaggle credentials below (from kaggle.com -> Account -> Create New Token):")
            k_user = input("Enter Kaggle Username: ").strip()
            k_key = input("Enter Kaggle Key: ").strip()
            os.environ['KAGGLE_USERNAME'] = k_user
            os.environ['KAGGLE_KEY'] = k_key
            
        os.makedirs(DATASET_DIR, exist_ok=True)
        !pip install kaggle -q
        !kaggle competitions download -c aptos2019-blindness-detection -p /content
        !unzip -q /content/aptos2019-blindness-detection.zip -d /content/dataset
        print("✅ Dataset extracted successfully!")

TRAIN_CSV = os.path.join(DATASET_DIR, 'train.csv')
TRAIN_IMAGES_DIR = os.path.join(DATASET_DIR, 'train_images')

df = pd.read_csv(TRAIN_CSV)
print(f"\n📊 Loaded {len(df)} training scans from {TRAIN_CSV}")
print("Class Distribution:")
class_names = ['0 - No DR', '1 - Mild NPDR', '2 - Moderate NPDR', '3 - Severe NPDR', '4 - Proliferative DR']
for c_idx, count in df['diagnosis'].value_counts().sort_index().items():
    print(f"   • Grade {c_idx} ({class_names[c_idx]}): {count} images ({count/len(df)*100:.1f}%)")

--- 
## 3. Ben Graham Preprocessing & PyTorch Dataset Loader

Applies the gold-standard **Ben Graham color subtraction algorithm**:
1. Removes retinal outer boundary artifacts via circular mask
2. Subtracts local Gaussian smoothed background ($30\times$ kernel scale)
3. Standardizes input tensor resolution to $512 \times 512$

In [ ]:
def crop_to_circle(img, tol=7):
    """Crops circular retinal boundary."""
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    if not mask.any():
        return img
    return img[np.ix_(mask.any(1), mask.any(0))]


def apply_ben_graham(img, target_size=512):
    """Ben Graham local color subtraction."""
    cropped = crop_to_circle(img)
    resized = cv2.resize(cropped, (target_size, target_size), interpolation=cv2.INTER_AREA)
    sigma = target_size / 30.0
    blurred = cv2.GaussianBlur(resized, (0, 0), sigma)
    enhanced = cv2.addWeighted(resized, 4.0, blurred, -4.0, 128)
    
    # Circular vignette mask
    mask = np.zeros((target_size, target_size), dtype=np.uint8)
    cv2.circle(mask, (target_size // 2, target_size // 2), int(target_size * 0.48), 255, -1)
    return cv2.bitwise_and(enhanced, enhanced, mask=mask)


class APTOSDataset(Dataset):
    def __init__(self, image_paths, labels, is_training=True, target_size=512):
        self.image_paths = image_paths
        self.labels = labels
        self.is_training = is_training
        self.target_size = target_size
        self.mean = np.array([0.485, 0.456, 0.406]).reshape(3, 1, 1).astype(np.float32)
        self.std = np.array([0.229, 0.224, 0.225]).reshape(3, 1, 1).astype(np.float32)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        bgr = cv2.imread(path)
        if bgr is None:
            bgr = np.zeros((self.target_size, self.target_size, 3), dtype=np.uint8)
            
        proc_bgr = apply_ben_graham(bgr, target_size=self.target_size)
        rgb = cv2.cvtColor(proc_bgr, cv2.COLOR_BGR2RGB)
        tensor = rgb.astype(np.float32) / 255.0

        # Data augmentation
        if self.is_training:
            if random.random() > 0.5: tensor = np.fliplr(tensor).copy()
            if random.random() > 0.5: tensor = np.flipud(tensor).copy()
            k = random.choice([0, 1, 2, 3])
            if k > 0: tensor = np.rot90(tensor, k).copy()

        tensor = np.transpose(tensor, (2, 0, 1))
        tensor = (tensor - self.mean) / self.std

        return torch.tensor(tensor, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


# Build train/val splits
all_paths = [os.path.join(TRAIN_IMAGES_DIR, f"{id_code}.png") for id_code in df['id_code']]
all_labels = df['diagnosis'].values

# 85% Train / 15% Validation (Stratified)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.15, stratify=all_labels, random_state=SEED
)

train_loader = DataLoader(APTOSDataset(train_paths, train_labels, is_training=True), batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(APTOSDataset(val_paths, val_labels, is_training=False), batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Training Loader: {len(train_paths)} samples | Validation Loader: {len(val_paths)} samples")

--- 
## 4. EfficientNet-B3 Deep Learning Classifier Architecture

In [ ]:
class NetraAIGradingModel(nn.Module):
    """
    EfficientNet-B3 Backbone with Custom Multi-Layer Classification Head
    and Dropout Regularization.
    """
    def __init__(self, num_classes=5):
        super().__init__()
        self.backbone = models.efficientnet_b3(weights=models.EfficientNet_B3_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features
        
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4),
            nn.Linear(in_features, 256),
            nn.SiLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(p=0.2),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)

model = NetraAIGradingModel().to(device)
print(f"✅ EfficientNet-B3 instantiated with {sum(p.numel() for p in model.parameters()):,} parameters.")

--- 
## 5. Cost-Sensitive Loss & AMP Scaler Configuration

In [ ]:
# Balanced inverse class weights
counts = np.bincount(train_labels, minlength=5)
weights = 1.0 / (counts.astype(np.float32) + 1e-5)
weights = torch.tensor(weights / weights.sum() * 5.0, dtype=torch.float32).to(device)
print("Class weights:", [round(w.item(), 3) for w in weights])

criterion = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
EPOCHS = 12
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler('cuda')

--- 
## 6. Training & Validation Execution (~10–12 Minutes)

In [ ]:
best_qwk = -1.0
save_path = os.path.join(OUTPUT_DIR, 'grading_efficientnet_b3.pt')

print("=" * 75)
print(f"   STARTING APTOS 2019 EFFICIENTNET-B3 TRAINING ON {device.type.upper()}")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad()
        
        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
            loss = criterion(outputs, targets)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * imgs.size(0)
        
    scheduler.step()
    epoch_train_loss = running_loss / len(train_paths)
    
    # Validation phase
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast('cuda'):
                outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(targets.numpy())
            
    val_preds = np.array(val_preds)
    val_targets = np.array(val_targets)
    
    qwk = cohen_kappa_score(val_targets, val_preds, weights='quadratic')
    acc = np.mean(val_targets == val_preds)
    
    # Referable DR metrics (Grade >= 2)
    b_true = (val_targets >= 2).astype(int)
    b_pred = (val_preds >= 2).astype(int)
    tn, fp, fn, tp = confusion_matrix(b_true, b_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    is_best = qwk > best_qwk
    if is_best:
        best_qwk = qwk
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_qwk': best_qwk,
            'backbone': 'efficientnet_b3',
            'accuracy': float(acc),
            'sensitivity': float(sens),
            'specificity': float(spec)
        }, save_path)
        
    print(f"Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {epoch_train_loss:.4f} | Acc: {acc*100:.1f}% | QWK: {qwk:.4f} | Ref Sens: {sens*100:.1f}% | Spec: {spec*100:.1f}% {'⭐ (New Best!)' if is_best else ''}")

print("=" * 75)
print(f"🎉 Training Complete! Peak Quadratic Weighted Kappa (QWK): {best_qwk:.4f}")
print(f"💾 Best Checkpoint saved to: {save_path}")

--- 
## 7. Clinical Confusion Matrix & Validation Audit

In [ ]:
# Load best checkpoint and evaluate
ckpt = torch.load(save_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

final_preds, final_targets = [], []
with torch.no_grad():
    for imgs, targets in val_loader:
        imgs = imgs.to(device)
        with torch.amp.autocast('cuda'):
            outputs = model(imgs)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        final_preds.extend(preds)
        final_targets.extend(targets.numpy())

cm = confusion_matrix(final_targets, final_preds, labels=[0, 1, 2, 3, 4])
print("\n" + "=" * 65)
print("           CONFUSION MATRIX (Actual vs. AI Predicted)")
print("=" * 65)
print("Actual \\ Pred  | Grade 0  | Grade 1  | Grade 2  | Grade 3  | Grade 4")
print("-" * 65)
for i, row in enumerate(cm):
    print(f"Grade {i} ({class_names[i][4:12]:<8}) | " + " | ".join(f"{val:<8}" for val in row))
print("=" * 65)

print("\nDetailed Classification Report:")
print(classification_report(final_targets, final_preds, target_names=class_names, digits=3))

--- 
## 8. Download Checkpoint to Laptop

Download `grading_efficientnet_b3.pt` and place it in your local project folder:
📁 `c:\Users\LENONO\Desktop\SIH 2026\SIH26038\ml\checkpoints\grading_efficientnet_b3.pt`

In [ ]:
if IS_KAGGLE:
    print("🎉 Running on Kaggle! Your model is saved at:")
    print(f"   {save_path}")
    print("To download: In the right sidebar, open 'Output' -> Click the 3 dots next to 'grading_efficientnet_b3.pt' -> 'Download'!")
else:
    from google.colab import files
    print("📥 Downloading checkpoint to your browser...")
    files.download(save_path)